# Automated Visual Defect Detection with CNNs
## Manufacturing Quality Control - Casting Products

This notebook implements a CNN-based visual inspection system for defective casting products, progressing from baseline to improved models.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 1. Data Preparation & Exploration

In [ ]:
# Data loading from Kaggle dataset structure
data_dir = 'archive/casting_512x512/casting_512x512'

# Verify dataset exists
if os.path.exists(data_dir):
    print(f"Dataset found at {data_dir}")
    subdirs = os.listdir(data_dir)
    print(f"Subdirectories: {subdirs}")
else:
    print(f"Dataset not found at {data_dir}")
    print("Please download the dataset from: https://www.kaggle.com/datasets/ravirajsingh45/real-life-industrial-dataset-of-casting-product")

In [ ]:
# Load and organize data
from pathlib import Path
from PIL import Image

IMG_SIZE = 224  # Downscale from 512 to keep memory manageable on CPU

images = []
labels = []

# Define class directories
def_front_dir = os.path.join(data_dir, 'def_front')
ok_front_dir = os.path.join(data_dir, 'ok_front')

def load_image(img_path):
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.float32) / 255.0  # Normalize (float32, not float64)

# Load defective images (label: 1)
if os.path.exists(def_front_dir):
    for img_file in os.listdir(def_front_dir)[:500]:  # Limit for computational efficiency
        try:
            images.append(load_image(os.path.join(def_front_dir, img_file)))
            labels.append(1)  # Defective
        except Exception as e:
            print(f"Error loading {img_file}: {e}")

# Load ok images (label: 0)
if os.path.exists(ok_front_dir):
    for img_file in os.listdir(ok_front_dir)[:500]:  # Limit for computational efficiency
        try:
            images.append(load_image(os.path.join(ok_front_dir, img_file)))
            labels.append(0)  # OK
        except Exception as e:
            print(f"Error loading {img_file}: {e}")

X = np.stack(images)
y = np.array(labels)
del images

print(f"Dataset loaded: {X.shape[0]} images")
print(f"Image shape: {X[0].shape}")
print(f"Memory: {X.nbytes / 1e9:.2f} GB")
print(f"Class distribution - OK: {np.sum(y==0)}, Defective: {np.sum(y==1)}")
print(f"Class imbalance ratio: {np.sum(y==1)/np.sum(y==0):.2f}")

In [ ]:
# Visualize sample images from both classes
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Images: Top row (OK), Bottom row (Defective)', fontsize=14)

# OK samples
ok_indices = np.where(y == 0)[0][:5]
for i, idx in enumerate(ok_indices):
    axes[0, i].imshow(X[idx])
    axes[0, i].set_title('OK')
    axes[0, i].axis('off')

# Defective samples
def_indices = np.where(y == 1)[0][:5]
for i, idx in enumerate(def_indices):
    axes[1, i].imshow(X[idx])
    axes[1, i].set_title('Defective')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print("Sample images visualization complete")

In [ ]:
# Split data: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
del X_temp, y_temp  # Free the intermediate copy

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTrain class distribution - OK: {np.sum(y_train==0)}, Defective: {np.sum(y_train==1)}")
print(f"Test class distribution - OK: {np.sum(y_test==0)}, Defective: {np.sum(y_test==1)}")

## 2. Phase 1: Baseline CNN Classifier

In [ ]:
# Build baseline CNN from scratch
def create_baseline_model(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # First convolutional block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Second convolutional block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Third convolutional block
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Global pooling instead of Flatten: avoids a ~67M-param Dense layer
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    
    return model

baseline_model = create_baseline_model()
baseline_model.summary()

In [ ]:
# Compile baseline model
baseline_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

# Train baseline model
print("Training Baseline Model...")
baseline_history = baseline_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Evaluate baseline on test set
y_pred_baseline_prob = baseline_model.predict(X_test)
y_pred_baseline = (y_pred_baseline_prob > 0.5).astype(int).flatten()

# Calculate metrics
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_precision = precision_score(y_test, y_pred_baseline)
baseline_recall = recall_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)
baseline_cm = confusion_matrix(y_test, y_pred_baseline)

print("\n" + "="*50)
print("BASELINE MODEL - Test Set Performance")
print("="*50)
print(f"Accuracy:  {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1-Score:  {baseline_f1:.4f}")
print(f"\nConfusion Matrix:")
print(baseline_cm)
print(f"\nTrue Negatives:  {baseline_cm[0,0]}")
print(f"False Positives: {baseline_cm[0,1]}")
print(f"False Negatives: {baseline_cm[1,0]}")
print(f"True Positives:  {baseline_cm[1,1]}")

In [ ]:
# Visualize baseline confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training history
axes[0].plot(baseline_history.history['loss'], label='Training Loss')
axes[0].plot(baseline_history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Baseline Model - Training History')
axes[0].legend()
axes[0].grid(True)

# Confusion matrix
sns.heatmap(baseline_cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], 
            xticklabels=['OK', 'Defective'], yticklabels=['OK', 'Defective'])
axes[1].set_title('Baseline Model - Confusion Matrix (Test Set)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Phase 1 Reflection

### Key Lessons from Baseline Model

1. **Model Architecture**: The baseline 3-layer CNN with progressive filter growth (32→64→128) provides a good foundation for feature extraction.

2. **Performance Insights**: 
   - The baseline achieves reasonable accuracy, but may have room for improvement
   - Precision vs Recall trade-off: In quality control, false negatives (missed defects) are more costly than false positives
   - The model learns reasonable patterns within the epochs shown

3. **Training Dynamics**:
   - Early stopping prevents overfitting
   - Learning rate reduction helps convergence in later epochs
   - The dropout layers are important for generalization

4. **Business Context**:
   - Minimizing false negatives (defects escaping to customers) is critical
   - Recall metric is more important than precision for this application

## 3. Phase 2: Improved CNN Model with Transfer Learning

In [ ]:
# Phase 2: Build improved model using Transfer Learning (MobileNetV2)
# Justification: 
# 1. Pre-trained on ImageNet captures general visual features
# 2. Reduces training time and data requirements
# 3. Better feature extraction for defect detection task
# 4. Fine-tuning allows adaptation to specific defect patterns

def create_improved_model(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    # Load pre-trained MobileNetV2 without top layers
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model weights initially
    base_model.trainable = False
    
    # Add custom top layers
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(2.0, offset=-1.0),  # [0,1] -> [-1,1], the range MobileNetV2 expects
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    
    return model, base_model

print("Creating improved model with transfer learning...")
improved_model, base_model = create_improved_model()
improved_model.summary()

In [ ]:
# Phase 2a: Train with frozen base model
improved_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Data augmentation for better generalization
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

print("Training improved model (phase 1 - frozen base)...")
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

improved_history_1 = improved_model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_val, y_val),
    epochs=20,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Phase 2b: Fine-tune with unfrozen base model layers
print("\nFine-tuning improved model (unfrozing base layers)...")

# Unfreeze last 50 layers of base model for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

# Recompile with lower learning rate
improved_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

improved_history_2 = improved_model.fit(
    train_datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_val, y_val),
    epochs=10,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Evaluate improved model on test set
y_pred_improved_prob = improved_model.predict(X_test)
y_pred_improved = (y_pred_improved_prob > 0.5).astype(int).flatten()

# Calculate metrics
improved_accuracy = accuracy_score(y_test, y_pred_improved)
improved_precision = precision_score(y_test, y_pred_improved)
improved_recall = recall_score(y_test, y_pred_improved)
improved_f1 = f1_score(y_test, y_pred_improved)
improved_cm = confusion_matrix(y_test, y_pred_improved)

print("\n" + "="*50)
print("IMPROVED MODEL - Test Set Performance")
print("="*50)
print(f"Accuracy:  {improved_accuracy:.4f}")
print(f"Precision: {improved_precision:.4f}")
print(f"Recall:    {improved_recall:.4f}")
print(f"F1-Score:  {improved_f1:.4f}")
print(f"\nConfusion Matrix:")
print(improved_cm)
print(f"\nTrue Negatives:  {improved_cm[0,0]}")
print(f"False Positives: {improved_cm[0,1]}")
print(f"False Negatives: {improved_cm[1,0]}")
print(f"True Positives:  {improved_cm[1,1]}")

## 4. Model Comparison & Analysis

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Baseline CNN': [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1],
    'Improved (Transfer Learning)': [improved_accuracy, improved_precision, improved_recall, improved_f1]
})

comparison_df['Improvement'] = comparison_df['Improved (Transfer Learning)'] - comparison_df['Baseline CNN']
comparison_df['% Change'] = (comparison_df['Improvement'] / comparison_df['Baseline CNN'] * 100).round(2)

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Baseline vs Improved Model Comparison', fontsize=16, fontweight='bold')

# Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
baseline_scores = [baseline_accuracy, baseline_precision, baseline_recall, baseline_f1]
improved_scores = [improved_accuracy, improved_precision, improved_recall, improved_f1]

x = np.arange(len(metrics))
width = 0.35

axes[0, 0].bar(x - width/2, baseline_scores, width, label='Baseline', alpha=0.8)
axes[0, 0].bar(x + width/2, improved_scores, width, label='Improved', alpha=0.8)
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Performance Metrics Comparison')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend()
axes[0, 0].set_ylim([0, 1])
axes[0, 0].grid(True, alpha=0.3)

# Confusion matrices side by side
sns.heatmap(baseline_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['OK', 'Defective'], yticklabels=['OK', 'Defective'])
axes[0, 1].set_title('Baseline - Confusion Matrix')
axes[0, 1].set_ylabel('True Label')
axes[0, 1].set_xlabel('Predicted Label')

sns.heatmap(improved_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1, 0],
            xticklabels=['OK', 'Defective'], yticklabels=['OK', 'Defective'])
axes[1, 0].set_title('Improved - Confusion Matrix')
axes[1, 0].set_ylabel('True Label')
axes[1, 0].set_xlabel('Predicted Label')

# Error analysis
baseline_false_neg = baseline_cm[1, 0]
improved_false_neg = improved_cm[1, 0]
baseline_false_pos = baseline_cm[0, 1]
improved_false_pos = improved_cm[0, 1]

error_types = ['False Negatives\n(Missed Defects)', 'False Positives\n(False Alarms)']
baseline_errors = [baseline_false_neg, baseline_false_pos]
improved_errors = [improved_false_neg, improved_false_pos]

x_err = np.arange(len(error_types))
axes[1, 1].bar(x_err - width/2, baseline_errors, width, label='Baseline', alpha=0.8)
axes[1, 1].bar(x_err + width/2, improved_errors, width, label='Improved', alpha=0.8)
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Error Analysis (Lower is Better)')
axes[1, 1].set_xticks(x_err)
axes[1, 1].set_xticklabels(error_types)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nFalse Negatives Reduction: {baseline_false_neg} → {improved_false_neg} (saved {baseline_false_neg - improved_false_neg} defects)")
print(f"False Positives Reduction: {baseline_false_pos} → {improved_false_pos}")

## Phase 2 Reflection

### Why Transfer Learning Improves Performance

1. **Pre-trained Feature Extraction**: MobileNetV2 learned visual patterns from 1.4M ImageNet images, providing strong initial features for object detection, edges, textures, and shapes.

2. **Better Generalization**: Transfer learning reduces overfitting by starting with a model that generalizes well, requiring less data to achieve high accuracy.

3. **Faster Convergence**: The model trains faster by fine-tuning rather than learning from scratch, reaching optimal weights in fewer epochs.

4. **Improved Recall**: Transfer learning helps identify subtle defect patterns that a shallow baseline CNN might miss, critical for quality control where missed defects are costly.

5. **Architecture Benefits**: 
   - Global average pooling reduces spatial dimensions more effectively
   - Batch normalization stabilizes training
   - Progressive dropout (0.4 → 0.3) provides graduated regularization

### Business Impact

- **Critical Metric - Recall**: The improved model better catches defective products before they reach customers
- **Cost Benefit**: Fewer false negatives significantly reduce warranty claims and reputation damage
- **Manufacturing Line**: Can be deployed on production line with confidence in real-time defect detection

In [ ]:
# Classification report for detailed analysis
print("\n" + "="*80)
print("BASELINE MODEL - DETAILED CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_pred_baseline, target_names=['OK', 'Defective']))

print("\n" + "="*80)
print("IMPROVED MODEL - DETAILED CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_test, y_pred_improved, target_names=['OK', 'Defective']))

## 5. Key Insights & Learning Journey

### Technical Lessons

1. **CNN Architecture**: Simple 3-layer CNNs work well for visual classification but may saturate quickly

2. **Transfer Learning Power**: Pre-trained models provide significant benefits for domain-specific problems like defect detection

3. **Data Augmentation**: Rotation, shifts, and zoom help the model generalize to variations in product orientation and lighting

4. **Regularization Strategy**: Dropout and batch normalization are critical for preventing overfitting in deep networks

5. **Optimization**: Fine-tuning lower learning rates during transfer learning prevents catastrophic forgetting

### Business Context

1. **Recall is Critical**: In manufacturing quality control, missing a defect is far more costly than a false alarm

2. **Automation Benefits**: Automated inspection reduces labor costs and provides consistency 24/7

3. **Scalability**: Once deployed, the model can process thousands of units per hour on production lines

4. **Continuous Improvement**: Logging predictions and collecting hard misclassifications allows periodic model retraining

### Future Enhancements

- Implement confidence thresholds for uncertain predictions (send to manual inspection)
- Deploy on edge devices (Raspberry Pi, NVIDIA Jetson) for line-side inference
- Track model performance metrics over time to detect model drift
- Collect additional defect images to expand training data and improve robustness